In [ ]:

from pathlib import Path

LEGALIR_SOURCE_PATH = Path("/kaggle/input/datasets/mduy2911/legaluit/LegalIR/train.json")
CORPUS_PATH = Path("/kaggle/input/datasets/mduy2911/legaluit/LegalIR/selected-contexts")
DENSE_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/modeluit/bge-m3-kaggle")
RERANKER_MODEL_PATH = Path("/kaggle/input/datasets/mduy2911/modeluit/bge-reranker-v2-m3-kaggle")
BM25_WHEEL_PATH = Path("/kaggle/input/datasets/mduy2911/offline-packages/bm25s-0.3.11-py3-none-any.whl")

DENSE_MODEL_NAME = "BAAI/bge-m3"
DENSE_DECLARED_REVISION = "5617a9f61b028005a4858fdac845db406aefb181"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANKER_DECLARED_REVISION = "953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e"
MODEL_PARAMETER_LIMIT = 4_000_000_000
EXPECTED_SOURCE_SHA256 = "c39cde9e74977e350f1456e7d487aafe67d2bcbaa4fa26fcabd557fe635635b7"
EXPECTED_SPLIT_COUNTS = {"train": 4_941, "dev": 1_036, "holdout": 1_023}
EXPECTED_DOCUMENTS = 8_532
EXPECTED_FIXED_CHUNKS = 199_816
CHUNK_SIZE, CHUNK_OVERLAP, CHUNK_STEP = 2_000, 200, 1_800
TOP_K_CHUNKS, CANDIDATE_DEPTH, SUPPORT_POOL_SIZE, FINAL_K = 2_000, 100, 8, 5
DENSE_MAX_LENGTH = RERANKER_MAX_SEQUENCE_LENGTH = 8_192
CORPUS_BATCH_SIZE, QUERY_BATCH_SIZE, RERANKER_BATCH_SIZE = 256, 64, 128
BOOTSTRAP_SEED, BOOTSTRAP_RESAMPLES = 20_260_913, 10_000
BASELINE_TOLERANCE = 1e-3
RRF_CONSTANT = 60
DENSE_BASELINE = {
    "recall_at_10": 0.9227799227799228, "recall_at_20": 0.9497265122265123,
    "recall_at_50": 0.9769144144144144, "recall_at_100": 0.9819015444015444,
    "mrr": 0.7159583402955311,
}
M8_REFERENCE = {"precision": 0.19150579150579153, "recall": 0.8973616473616474, "mrr": 0.7622471799366903}
RESULT_PATH = Path("/kaggle/working/bm25_bge_hybrid_fusion_dev_results.json")


In [ ]:

import os, subprocess, sys
os.environ.update({"HF_HUB_DISABLE_TELEMETRY": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1", "CUDA_VISIBLE_DEVICES": "0"})
for path, label, directory in ((LEGALIR_SOURCE_PATH, "LegalIR source", False), (CORPUS_PATH, "corpus", True), (DENSE_MODEL_PATH, "BGE-M3 snapshot", True), (RERANKER_MODEL_PATH, "BGE reranker snapshot", True), (BM25_WHEEL_PATH, "bm25s wheel", False)):
    if not (path.is_dir() if directory else path.is_file()):
        raise FileNotFoundError(f"Attach {label} at {path}")
subprocess.check_call([sys.executable, "-m", "pip", "install", "--no-index", str(BM25_WHEEL_PATH)])


In [ ]:
# Standalone fixed-DEV LegalIR implementation. No repository runtime is imported.
import gc
import json
from collections import Counter, defaultdict
from hashlib import sha256
from math import isfinite
from statistics import median
from time import perf_counter

import numpy as np
import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer


def read_json(path: Path):
    try:
        with path.open(encoding="utf-8-sig") as stream:
            return json.load(stream)
    except json.JSONDecodeError as exc:
        raise ValueError(f"{path}: invalid JSON: {exc}") from exc


def load_fixed_dev(path: Path) -> tuple[dict, dict]:
    source_digest = sha256(path.read_bytes()).hexdigest()
    if source_digest != EXPECTED_SOURCE_SHA256:
        raise RuntimeError(
            f"LegalIR source SHA-256 mismatch: expected {EXPECTED_SOURCE_SHA256}, "
            f"got {source_digest}. Stop before reading samples."
        )
    value = read_json(path)
    if not isinstance(value, dict) or not all(isinstance(v, dict) for v in value.values()):
        raise ValueError(f"{path}: expected an object keyed by sample ID")
    samples = {str(sample_id): sample for sample_id, sample in value.items()}
    if len(samples) != len(value):
        raise ValueError("duplicate sample IDs after string canonicalization")

    split_counts = {"train": 0, "dev": 0, "holdout": 0}
    dev_ids = []
    for sample_id, sample in samples.items():
        question = sample.get("question")
        group_key = question if isinstance(question, str) else f"\0fallback-sample-id:{sample_id}"
        bucket = int(sha256(group_key.encode("utf-8")).hexdigest()[:8], 16) % 100
        partition = "train" if bucket < 70 else "dev" if bucket < 85 else "holdout"
        split_counts[partition] += 1
        if partition == "dev":
            dev_ids.append(sample_id)
    if split_counts != EXPECTED_SPLIT_COUNTS:
        raise RuntimeError(f"fixed split counts mismatch: {split_counts}; stop")

    dev_ids.sort()
    dev = {sample_id: samples[sample_id] for sample_id in dev_ids}
    del samples, value
    for sample_id, sample in dev.items():
        if not isinstance(sample.get("question"), str):
            raise TypeError(f"DEV sample {sample_id!r}: question must be a string")
        answer = sample.get("answer")
        if not isinstance(answer, list) or not answer:
            raise ValueError(f"DEV sample {sample_id!r}: expected a non-empty answer list")
        canonical = [str(document_id) for document_id in answer]
        if len(canonical) != len(set(canonical)):
            raise ValueError(f"DEV sample {sample_id!r}: answer contains duplicate IDs")
    evaluated_partition = "dev"
    if evaluated_partition != "dev" or len(dev) != EXPECTED_SPLIT_COUNTS["dev"]:
        raise RuntimeError("this notebook may evaluate fixed DEV only")
    return dev, {
        "version": "legalir_split_v1",
        "evaluated_partition": evaluated_partition,
        "queries": len(dev),
        "source_sha256": source_digest,
        "split_counts": split_counts,
        "selection_only": True,
        "holdout_metrics_computed": False,
    }


def load_corpus(path: Path) -> list[dict]:
    paths = sorted(item for item in path.rglob("*") if item.is_file() and item.suffix.lower() == ".json")
    if not paths:
        raise ValueError(f"{path}: corpus directory contains no JSON files")
    documents = []
    for json_path in paths:
        value = read_json(json_path)
        values = value if isinstance(value, list) else [value]
        if not all(isinstance(document, dict) for document in values):
            raise ValueError(f"{json_path}: expected document object(s)")
        documents.extend(values)
    document_ids = []
    for document in documents:
        if document.get("id") is None:
            raise ValueError("corpus document is missing a non-null ID")
        document_id = str(document["id"])
        if not isinstance(document.get("passage"), str):
            raise TypeError(f"document {document_id!r}: passage must be a string")
        document_ids.append(document_id)
    if len(document_ids) != len(set(document_ids)):
        raise ValueError("corpus contains duplicate document IDs")
    if len(documents) != EXPECTED_DOCUMENTS:
        raise RuntimeError(f"expected {EXPECTED_DOCUMENTS} documents, got {len(documents)}")
    return documents


def distribution(values: list[int]) -> dict:
    array = np.asarray(values, dtype=np.int64)
    if array.size == 0:
        return {"min": None, "median": None, "p95": None, "max": None}
    return {
        "min": int(array.min()),
        "median": float(np.median(array)),
        "p95": float(np.percentile(array, 95)),
        "max": int(array.max()),
    }


def validate_chunk_provenance(documents: list[dict], chunks: list[dict]) -> dict:
    source_by_id = {str(document["id"]): document["passage"] for document in documents}
    chunk_ids = [chunk["chunk_id"] for chunk in chunks]
    if len(chunk_ids) != len(set(chunk_ids)):
        raise RuntimeError("chunk IDs must be unique")
    intervals = defaultdict(list)
    for chunk in chunks:
        document_id = chunk["document_id"]
        source = source_by_id.get(document_id)
        if source is None:
            raise RuntimeError("chunk references an unknown document")
        start, end = chunk["char_start"], chunk["char_end"]
        if not (0 <= start < end <= len(source)) or chunk["text"] != source[start:end]:
            raise RuntimeError(f"chunk {chunk['chunk_id']!r}: exact provenance failed")
        intervals[document_id].append((start, end))
    for document_id, source in source_by_id.items():
        if not source:
            continue
        ordered = sorted(intervals[document_id])
        if not ordered or ordered[0][0] != 0:
            raise RuntimeError(f"document {document_id!r}: coverage does not start at zero")
        covered_end = 0
        for start, end in ordered:
            if start > covered_end:
                raise RuntimeError(f"document {document_id!r}: chunk coverage gap")
            covered_end = max(covered_end, end)
        if covered_end != len(source):
            raise RuntimeError(f"document {document_id!r}: incomplete source coverage")
    return {
        "exact_source_slices": True,
        "unique_chunk_ids": True,
        "complete_non_empty_source_coverage": True,
    }


def fixed_window_chunks(documents: list[dict], chunk_size: int, overlap: int) -> tuple[list[dict], dict]:
    if chunk_size <= 0 or overlap < 0 or overlap >= chunk_size:
        raise ValueError("invalid fixed-window parameters")
    step = chunk_size - overlap
    chunks = []
    counts = []
    for document in documents:
        document_id = str(document["id"])
        source = document["passage"]
        before = len(chunks)
        for chunk_index, start in enumerate(range(0, len(source), step)):
            end = min(start + chunk_size, len(source))
            chunks.append({
                "chunk_id": f"{document_id}:{chunk_index}",
                "document_id": document_id,
                "chunk_index": chunk_index,
                "text": source[start:end],
                "char_start": start,
                "char_end": end,
            })
            if end == len(source):
                break
        counts.append(len(chunks) - before)
    provenance = validate_chunk_provenance(documents, chunks)
    return chunks, {
        "number_of_chunks": len(chunks),
        "chunk_size": chunk_size,
        "overlap": overlap,
        "step": step,
        "chunk_length_characters": distribution([len(chunk["text"]) for chunk in chunks]),
        "chunks_per_document": distribution(counts),
        "provenance": provenance,
    }


def model_metadata(
    model, model_name: str, declared_revision: str, path: Path,
    trust_remote_code: bool = False,
) -> dict:
    import re
    if not re.fullmatch(r"[0-9a-f]{40}", declared_revision):
        raise RuntimeError(
            f"{model_name}: replace declared revision with the resolved 40-hex snapshot commit SHA"
        )
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    eligible = parameter_count < MODEL_PARAMETER_LIMIT
    print(
        "Model eligibility:\n"
        f"model = {model_name}\n"
        f"parameter_count = {parameter_count}\n"
        f"competition_limit = < {MODEL_PARAMETER_LIMIT:,}\n"
        f"eligible = {str(eligible).lower()}"
    )
    if not eligible:
        raise RuntimeError(f"{model_name} is ineligible: parameter_count >= 4,000,000,000")
    value = getattr(model.config, "_commit_hash", None)
    config_hash = value.strip() if isinstance(value, str) and value.strip() else None
    if config_hash is not None and config_hash != declared_revision:
        raise RuntimeError(
            f"{model_name} config revision {config_hash!r} != declared {declared_revision!r}"
        )
    return {
        "model_repository": model_name,
        "declared_revision": declared_revision,
        "resolved_snapshot": config_hash or declared_revision,
        "config_commit_hash": config_hash,
        "revision_status": "verified-from-config" if config_hash else "declared-offline-snapshot",
        "actual_parameter_count": int(parameter_count),
        "competition_parameter_limit_exclusive": MODEL_PARAMETER_LIMIT,
        "eligible_under_4b_rule": eligible,
        "local_path": str(path),
        "local_files_only": True,
        "trust_remote_code": trust_remote_code,
    }


def load_dense_model() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_PATH, local_files_only=True)
    model = AutoModel.from_pretrained(DENSE_MODEL_PATH, dtype=torch.float16, local_files_only=True)
    if int(getattr(model.config, "max_position_embeddings", 0)) < DENSE_MAX_LENGTH:
        raise RuntimeError("local dense model does not support max_length=8192")
    metadata = model_metadata(model, DENSE_MODEL_NAME, DENSE_DECLARED_REVISION, DENSE_MODEL_PATH)
    model.to("cuda").eval()
    return {"tokenizer": tokenizer, "model": model, "metadata": metadata, "load_seconds": perf_counter() - started}


def load_reranker() -> dict:
    if not torch.cuda.is_available():
        raise RuntimeError("Enable a Kaggle CUDA accelerator")
    started = perf_counter()
    tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_PATH, local_files_only=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        RERANKER_MODEL_PATH, dtype=torch.float16, local_files_only=True
    )
    if int(getattr(model.config, "max_position_embeddings", 0)) < RERANKER_MAX_SEQUENCE_LENGTH:
        raise RuntimeError("local reranker does not support max_sequence_length=8192")
    metadata = model_metadata(model, RERANKER_MODEL_NAME, RERANKER_DECLARED_REVISION, RERANKER_MODEL_PATH)
    model.to("cuda").eval()
    return {"tokenizer": tokenizer, "model": model, "metadata": metadata, "load_seconds": perf_counter() - started}


def encode_normalized_cls(model_bundle: dict, texts: list[str], batch_size: int) -> dict:
    embeddings = []
    started = perf_counter()
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        inputs = model_bundle["tokenizer"](
            batch, padding=True, truncation=True, max_length=DENSE_MAX_LENGTH, return_tensors="pt"
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        with torch.no_grad():
            output = model_bundle["model"](**inputs, return_dict=True)
            embedding = F.normalize(output.last_hidden_state[:, 0], p=2, dim=1)
        if embedding.ndim != 2 or not torch.isfinite(embedding).all():
            raise RuntimeError("dense encoder returned invalid CLS embeddings")
        embeddings.append(embedding.cpu())
    encoded = torch.cat(embeddings, dim=0)
    if encoded.shape[0] != len(texts):
        raise RuntimeError("dense embedding count mismatch")
    return {"embeddings": encoded, "seconds": perf_counter() - started}


def retrieve_dense_hits(query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor, chunks: list[dict], sample_ids: list[str]) -> dict:
    if query_embeddings.shape[0] != len(sample_ids) or corpus_embeddings.shape[0] != len(chunks):
        raise ValueError("embedding count mismatch")
    if len(chunks) < TOP_K_CHUNKS:
        raise ValueError("corpus has fewer chunks than requested retrieval depth")
    started = perf_counter()
    corpus_gpu = corpus_embeddings.to("cuda")
    hits = {}
    for start in range(0, len(sample_ids), QUERY_BATCH_SIZE):
        batch_ids = sample_ids[start:start + QUERY_BATCH_SIZE]
        query_gpu = query_embeddings[start:start + len(batch_ids)].to("cuda")
        similarities = query_gpu @ corpus_gpu.T
        if not torch.isfinite(similarities).all():
            raise RuntimeError("dense similarity contains non-finite values")
        scores, indices = torch.topk(similarities, k=TOP_K_CHUNKS, dim=1, sorted=True)
        for row, sample_id in enumerate(batch_ids):
            ordered = list(zip(scores[row].float().cpu().tolist(), indices[row].cpu().tolist()))
            ordered.sort(key=lambda item: (-item[0], item[1]))
            hits[sample_id] = [
                {
                    "chunk_index": int(chunk_index),
                    "document_id": chunks[int(chunk_index)]["document_id"],
                    "score": float(score),
                    "chunk_rank": rank,
                }
                for rank, (score, chunk_index) in enumerate(ordered, start=1)
            ]
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_gpu
    torch.cuda.empty_cache()
    return {"hits": hits, "seconds": seconds}


def dense_aggregate(scores: list[float], rule: str) -> float:
    ordered = sorted(scores, reverse=True)
    if rule == "max_top1":
        return ordered[0]
    if rule == "sum_top2":
        return sum(ordered[:2])
    if rule == "mean_top2":
        selected = ordered[:2]
        return sum(selected) / len(selected)
    if rule == "sum_top3":
        return sum(ordered[:3])
    raise ValueError(f"unknown dense aggregation rule: {rule}")


def candidate_depth_diagnostics(hits_by_query: dict, target_depth: int) -> dict:
    available = [len({hit["document_id"] for hit in hits}) for hits in hits_by_query.values()]
    effective = [min(target_depth, value) for value in available]
    return {
        "target_depth": target_depth,
        "available_unique_documents": distribution(available),
        "effective_candidate_depth": distribution(effective),
        "queries_below_target": sum(value < target_depth for value in available),
        "policy": "effective_depth=min(target_depth, available_unique_documents); never pad",
    }


def candidates_from_hits(
    hits_by_query: dict,
    rule: str,
    depth: int,
    support_limit: int = 8,
    require_exact_depth: bool = True,
) -> dict:
    output = {}
    for sample_id, hits in hits_by_query.items():
        grouped = defaultdict(list)
        for hit in hits:
            if not isfinite(hit["score"]):
                raise RuntimeError("non-finite dense hit")
            grouped[hit["document_id"]].append(hit)
        documents = []
        for document_id, document_hits in grouped.items():
            ordered = sorted(document_hits, key=lambda item: (-item["score"], item["chunk_rank"], item["chunk_index"]))
            documents.append({
                "document_id": document_id,
                "dense_document_score": dense_aggregate([item["score"] for item in ordered], rule),
                "best_chunk_rank": ordered[0]["chunk_rank"],
                "available_global_hits": len(ordered),
                "supporting_chunk_indices": [item["chunk_index"] for item in ordered[:support_limit]],
            })
        documents.sort(key=lambda item: (-item["dense_document_score"], item["best_chunk_rank"], item["document_id"]))
        selected = documents[:depth]
        if not selected:
            raise RuntimeError(f"sample {sample_id!r}: retrieval produced no candidates")
        if require_exact_depth and len(selected) != depth:
            raise RuntimeError(f"sample {sample_id!r}: expected {depth} candidates")
        for rank, document in enumerate(selected, start=1):
            document["original_dense_rank"] = rank
        ids = [document["document_id"] for document in selected]
        if len(ids) != len(set(ids)):
            raise RuntimeError("candidate ranking contains duplicate document IDs")
        output[sample_id] = selected
    return output


def rankings_from_candidates(candidates: dict) -> dict:
    return {
        sample_id: [document["document_id"] for document in documents]
        for sample_id, documents in candidates.items()
    }


def evaluate_rankings(samples: dict, rankings: dict, depths=(5, 10, 20, 50, 100)) -> dict:
    if set(samples) != set(rankings):
        raise RuntimeError("ranking IDs do not match fixed DEV IDs")
    recalls = {depth: [] for depth in depths}
    reciprocal_ranks = []
    precision_5, recall_5 = [], []
    for sample_id, sample in samples.items():
        ranked = [str(document_id) for document_id in rankings[sample_id]]
        if len(ranked) != len(set(ranked)):
            raise RuntimeError(f"sample {sample_id!r}: duplicate ranked document IDs")
        gold = {str(document_id) for document_id in sample["answer"]}
        for depth in depths:
            effective_k = min(depth, len(ranked))
            recalls[depth].append(len(gold.intersection(ranked[:effective_k])) / len(gold))
        first = next((rank for rank, document_id in enumerate(ranked, 1) if document_id in gold), None)
        reciprocal_ranks.append(0.0 if first is None else 1.0 / first)
        predicted = ranked[:FINAL_K]
        if not 1 <= len(predicted) <= 5:
            raise RuntimeError("prediction length must be between one and five")
        overlap = len(gold.intersection(predicted))
        precision_5.append(overlap / len(predicted))
        recall_5.append(overlap / len(gold))
    return {
        "precision": float(np.mean(precision_5)),
        "recall": float(np.mean(recall_5)),
        "mrr": float(np.mean(reciprocal_ranks)),
        **{f"recall_at_{depth}": float(np.mean(values)) for depth, values in recalls.items()},
    }


def first_gold_bins(samples: dict, rankings: dict) -> dict:
    bins = Counter()
    found = []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        first = next((rank for rank, document_id in enumerate(rankings[sample_id], 1) if document_id in gold), None)
        if first is None:
            bins["not_found"] += 1
            continue
        found.append(first)
        label = "1" if first == 1 else "2_5" if first <= 5 else "6_10" if first <= 10 else "11_20" if first <= 20 else "21_50" if first <= 50 else "51_100" if first <= 100 else "beyond_100"
        bins[label] += 1
    labels = ("1", "2_5", "6_10", "11_20", "21_50", "51_100", "beyond_100", "not_found")
    return {
        "counts": {label: bins[label] for label in labels},
        "median_when_found": float(median(found)) if found else None,
    }


def top5_contributions(samples: dict, rankings: dict) -> dict:
    precision, recall = [], []
    for sample_id, sample in samples.items():
        gold = {str(document_id) for document_id in sample["answer"]}
        predicted = rankings[sample_id][:FINAL_K]
        overlap = len(gold.intersection(predicted))
        precision.append(overlap / len(predicted))
        recall.append(overlap / len(gold))
    return {"precision": np.asarray(precision), "recall": np.asarray(recall)}


def paired_behavior(control: dict, alternative: dict) -> dict:
    result = {}
    for metric in ("precision", "recall"):
        delta = alternative[metric] - control[metric]
        result[metric] = {
            "improved": int(np.sum(delta > 0)),
            "unchanged": int(np.sum(delta == 0)),
            "worsened": int(np.sum(delta < 0)),
        }
    return result


def paired_bootstrap(control: dict, alternative: dict) -> dict:
    rng = np.random.default_rng(BOOTSTRAP_SEED)
    result = {"seed": BOOTSTRAP_SEED, "resamples": BOOTSTRAP_RESAMPLES}
    for metric in ("precision", "recall"):
        delta = alternative[metric] - control[metric]
        values = np.empty(BOOTSTRAP_RESAMPLES, dtype=np.float64)
        for start in range(0, BOOTSTRAP_RESAMPLES, 512):
            stop = min(start + 512, BOOTSTRAP_RESAMPLES)
            indices = rng.integers(0, len(delta), size=(stop - start, len(delta)))
            values[start:stop] = delta[indices].mean(axis=1)
        result[metric] = {
            "observed_delta": float(delta.mean()),
            "percentile_interval_95": [float(np.percentile(values, 2.5)), float(np.percentile(values, 97.5))],
            "fraction_delta_gt_0": float(np.mean(values > 0)),
        }
    return result


def score_pair_records(reranker: dict, records: list[tuple[tuple, str, str]], collect_token_lengths: bool = False) -> dict:
    cache = {}
    token_lengths = []
    forward_seconds = 0.0
    started = perf_counter()
    for start in range(0, len(records), RERANKER_BATCH_SIZE):
        batch = records[start:start + RERANKER_BATCH_SIZE]
        questions = [record[1] for record in batch]
        passages = [record[2] for record in batch]
        if collect_token_lengths:
            unt = reranker["tokenizer"](questions, passages, padding=False, truncation=False, return_length=True)
            token_lengths.extend(int(length) for length in unt["length"])
        inputs = reranker["tokenizer"](
            questions, passages, padding=True, truncation="only_second",
            max_length=RERANKER_MAX_SEQUENCE_LENGTH, return_tensors="pt",
        )
        inputs = {name: value.to("cuda") for name, value in inputs.items()}
        torch.cuda.synchronize()
        forward_started = perf_counter()
        with torch.no_grad():
            logits = reranker["model"](**inputs, return_dict=True).logits.view(-1).float()
        torch.cuda.synchronize()
        forward_seconds += perf_counter() - forward_started
        scores = logits.cpu().tolist()
        if len(scores) != len(batch) or not all(isfinite(score) for score in scores):
            raise RuntimeError("reranker returned invalid scores")
        for record, score in zip(batch, scores):
            if record[0] in cache:
                raise RuntimeError("duplicate CE pair key")
            cache[record[0]] = float(score)
    diagnostics = {
        "pairs": len(records),
        "forward_seconds": forward_seconds,
        "total_seconds": perf_counter() - started,
    }
    if collect_token_lengths:
        lengths = np.asarray(token_lengths, dtype=np.int64)
        diagnostics["token_length_before_truncation"] = {
            "min": int(lengths.min()),
            "median": float(np.median(lengths)),
            "p95": float(np.percentile(lengths, 95)),
            "max": int(lengths.max()),
        }
        diagnostics["truncation_count"] = int(np.sum(lengths > RERANKER_MAX_SEQUENCE_LENGTH))
    return {"cache": cache, "diagnostics": diagnostics}


def support_map_from_candidates(candidates: dict) -> dict:
    return {
        sample_id: {
            document["document_id"]: list(document["supporting_chunk_indices"])
            for document in documents
        }
        for sample_id, documents in candidates.items()
    }


def make_ce_records(samples: dict, candidates: dict, supports: dict, chunks: list[dict], text_builder=None, key_prefix="ce") -> list[tuple]:
    records = []
    for sample_id, sample in samples.items():
        for document in candidates[sample_id]:
            document_id = document["document_id"]
            indices = supports[sample_id][document_id]
            if not 1 <= len(indices) <= SUPPORT_POOL_SIZE:
                raise RuntimeError("support pool must contain one to eight chunks")
            for support_position, chunk_index in enumerate(indices):
                text = chunks[chunk_index]["text"] if text_builder is None else text_builder(document_id, chunk_index)
                key = (key_prefix, sample_id, document_id, chunk_index)
                records.append((key, sample["question"], text))
    return records


def ce_aggregate(ordered_scores: list[float], rule: str) -> float:
    if rule == "max_top1":
        return ordered_scores[0]
    if rule == "sum_top2":
        return sum(ordered_scores[:2])
    if rule == "mean_top2":
        selected = ordered_scores[:2]
        return sum(selected) / len(selected)
    if rule == "sum_top3":
        return sum(ordered_scores[:3])
    raise ValueError(f"unknown CE aggregation rule: {rule}")


def derive_ce_ranking(samples: dict, candidates: dict, supports: dict, score_cache: dict, rule="sum_top2", key_prefix="ce") -> dict:
    rankings = {}
    for sample_id in samples:
        scored_documents = []
        for original_rank, document in enumerate(candidates[sample_id], start=1):
            document_id = document["document_id"]
            scored_chunks = []
            for support_position, chunk_index in enumerate(supports[sample_id][document_id]):
                key = (key_prefix, sample_id, document_id, chunk_index)
                if key not in score_cache:
                    raise RuntimeError("missing CE score; no imputation is allowed")
                scored_chunks.append((score_cache[key], support_position, chunk_index))
            scored_chunks.sort(key=lambda item: (-item[0], item[1], item[2]))
            document_score = ce_aggregate([item[0] for item in scored_chunks], rule)
            if not isfinite(document_score):
                raise RuntimeError("non-finite CE document score")
            scored_documents.append((document_id, document_score, original_rank))
        scored_documents.sort(key=lambda item: (-item[1], item[2], item[0]))
        ranking = [item[0] for item in scored_documents]
        expected = [document["document_id"] for document in candidates[sample_id]]
        if len(ranking) != len(set(ranking)) or set(ranking) != set(expected):
            raise RuntimeError("CE reranking changed the candidate set")
        rankings[sample_id] = ranking
    return rankings


def document_chunk_indices(chunks: list[dict]) -> dict:
    output = defaultdict(list)
    for chunk_index, chunk in enumerate(chunks):
        output[chunk["document_id"]].append(chunk_index)
    return dict(output)


def full_document_supports(query_embeddings: torch.Tensor, corpus_embeddings: torch.Tensor, chunks: list[dict], sample_ids: list[str], candidates: dict) -> dict:
    by_document = document_chunk_indices(chunks)
    corpus_gpu = corpus_embeddings.to("cuda")
    supports = {}
    started = perf_counter()
    for query_index, sample_id in enumerate(sample_ids):
        query = query_embeddings[query_index:query_index + 1].to("cuda")
        supports[sample_id] = {}
        for document in candidates[sample_id]:
            document_id = document["document_id"]
            indices = by_document.get(document_id, [])
            if not indices:
                raise RuntimeError("candidate document has no fixed-window chunks")
            index_tensor = torch.tensor(indices, device="cuda", dtype=torch.long)
            scores = (query @ corpus_gpu.index_select(0, index_tensor).T).view(-1)
            if not torch.isfinite(scores).all():
                raise RuntimeError("within-document dense similarity is non-finite")
            ordered = list(zip(scores.float().cpu().tolist(), indices))
            ordered.sort(key=lambda item: (-item[0], item[1]))
            supports[sample_id][document_id] = [item[1] for item in ordered[:SUPPORT_POOL_SIZE]]
    torch.cuda.synchronize()
    seconds = perf_counter() - started
    del corpus_gpu
    torch.cuda.empty_cache()
    return {"supports": supports, "seconds": seconds}


def metric_delta(alternative: dict, control: dict) -> dict:
    return {key: alternative[key] - control[key] for key in control if isinstance(control[key], float)}


def assert_dense_baseline(metrics: dict) -> None:
    observed = {key: metrics[key] for key in DENSE_BASELINE}
    delta = {key: observed[key] - DENSE_BASELINE[key] for key in DENSE_BASELINE}
    if max(abs(value) for value in delta.values()) > BASELINE_TOLERANCE:
        raise RuntimeError(f"fixed-window dense control mismatch: {delta}; stop before variants")


def assert_m8_control(metrics: dict) -> None:
    expected = {"precision": M8_REFERENCE["precision"], "recall": M8_REFERENCE["recall"], "mrr": M8_REFERENCE["mrr"]}
    delta = {key: metrics[key] - expected[key] for key in expected}
    if max(abs(value) for value in delta.values()) > BASELINE_TOLERANCE:
        raise RuntimeError(f"m8 control mismatch: {delta}; stop before variants")


def save_result(result: dict) -> None:
    if result["split"]["evaluated_partition"] != "dev":
        raise RuntimeError("refusing to write a non-DEV result")
    RESULT_PATH.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(result, ensure_ascii=False, indent=2))
    print("Saved aggregate-only DEV result:", RESULT_PATH)

import bm25s
import re

BM25_TOKEN_PATTERN = re.compile(r"\w+", flags=re.UNICODE)


def build_bm25_rankings(chunks: list[dict], samples: dict) -> dict:
    tokenized_chunks = bm25s.tokenize(
        [chunk["text"] for chunk in chunks], lower=True, token_pattern=r"(?u)\w+",
        stopwords=[], stemmer=None, return_ids=True, show_progress=False,
    )
    retriever = bm25s.BM25(k1=1.5, b=0.75, method="lucene")
    retriever.index(tokenized_chunks, show_progress=False)
    output = {}
    sample_items = list(samples.items())
    for start in range(0, len(sample_items), QUERY_BATCH_SIZE):
        batch = sample_items[start:start + QUERY_BATCH_SIZE]
        query_tokens = [BM25_TOKEN_PATTERN.findall(sample["question"].lower()) for _, sample in batch]
        result = retriever.retrieve(query_tokens, k=TOP_K_CHUNKS, sorted=True, return_as="tuple", show_progress=False)
        for (sample_id, _), indices, scores in zip(batch, result.documents, result.scores):
            hits = []
            ordered = [(int(chunk_index), float(score)) for chunk_index, score in zip(indices, scores)]
            ordered.sort(key=lambda item: (-item[1], item[0]))
            for rank, (chunk_index, value) in enumerate(ordered, start=1):
                chunk_index = int(chunk_index)
                if not isfinite(value):
                    raise RuntimeError("BM25 returned a non-finite score")
                hits.append({
                    "chunk_index": chunk_index,
                    "document_id": chunks[chunk_index]["document_id"],
                    "score": value,
                    "chunk_rank": rank,
                })
            output[str(sample_id)] = candidates_from_hits({str(sample_id): hits}, "sum_top2", 100)[str(sample_id)]
    return output



def retrieve_bm25_hits(chunks: list[dict], samples: dict) -> dict:
    tokenized = bm25s.tokenize([chunk["text"] for chunk in chunks], lower=True, token_pattern=r"(?u)\w+", stopwords=[], stemmer=None, return_ids=True, show_progress=False)
    index = bm25s.BM25(k1=1.5, b=0.75, method="lucene")
    index.index(tokenized, show_progress=False)
    output = {}
    started = perf_counter()
    items = list(samples.items())
    for start in range(0, len(items), QUERY_BATCH_SIZE):
        batch = items[start:start + QUERY_BATCH_SIZE]
        queries = [BM25_TOKEN_PATTERN.findall(sample["question"].lower()) for _, sample in batch]
        result = index.retrieve(queries, k=TOP_K_CHUNKS, sorted=True, return_as="tuple", show_progress=False)
        for (sample_id, _), indices, scores in zip(batch, result.documents, result.scores):
            ordered = sorted(((float(score), int(idx)) for idx, score in zip(indices, scores)), key=lambda item: (-item[0], item[1]))
            output[sample_id] = [{"chunk_index": idx, "document_id": chunks[idx]["document_id"], "score": score, "chunk_rank": rank} for rank, (score, idx) in enumerate(ordered, 1)]
    return {"hits": output, "seconds": perf_counter() - started}


def fuse_hits(left: dict, right: dict, method: str) -> dict:
    fused = {}
    for sample_id in left:
        left_hits, right_hits = left[sample_id], right[sample_id]
        scores, best_rank = defaultdict(float), {}
        if method == "rrf":
            for branch in (left_hits, right_hits):
                for hit in branch:
                    idx = hit["chunk_index"]
                    scores[idx] += 1.0 / (RRF_CONSTANT + hit["chunk_rank"])
                    best_rank[idx] = min(best_rank.get(idx, TOP_K_CHUNKS + 1), hit["chunk_rank"])
        elif method == "normalized_score":
            for branch in (left_hits, right_hits):
                values = [hit["score"] for hit in branch]
                low, high = min(values), max(values)
                denominator = high - low
                for hit in branch:
                    value = 0.0 if denominator <= 1e-12 else (hit["score"] - low) / denominator
                    idx = hit["chunk_index"]
                    scores[idx] += 0.5 * value
                    best_rank[idx] = min(best_rank.get(idx, TOP_K_CHUNKS + 1), hit["chunk_rank"])
        else:
            raise ValueError(method)
        ordered = sorted(scores, key=lambda idx: (-scores[idx], best_rank[idx], idx))[:TOP_K_CHUNKS]
        fused[sample_id] = [{"chunk_index": idx, "document_id": chunks[idx]["document_id"], "score": float(scores[idx]), "chunk_rank": rank} for rank, idx in enumerate(ordered, 1)]
    return fused


def pool_gold_coverage(samples: dict, hits: dict) -> dict:
    values, query_any = [], []
    for sample_id, sample in samples.items():
        gold = {str(value) for value in sample["answer"]}
        present = {hit["document_id"] for hit in hits[sample_id]}
        overlap = gold & present
        values.append(len(overlap) / len(gold)); query_any.append(bool(overlap))
    return {"mean_gold_document_recall": float(np.mean(values)), "queries_with_any_gold": int(sum(query_any)), "queries": len(values)}


def complementarity(samples: dict, a: dict, b: dict, depth: int) -> dict:
    query_counts = Counter(); gold_counts = Counter()
    for sample_id, sample in samples.items():
        ga = set(a[sample_id][:depth]); gb = set(b[sample_id][:depth]); gold = {str(x) for x in sample["answer"]}
        qa, qb = bool(gold & ga), bool(gold & gb)
        query_counts["both" if qa and qb else "a_only" if qa else "b_only" if qb else "neither"] += 1
        for doc in gold:
            ia, ib = doc in ga, doc in gb
            gold_counts["both" if ia and ib else "a_only" if ia else "b_only" if ib else "neither"] += 1
    return {"depth": depth, "query_coverage": dict(query_counts), "gold_document_coverage": dict(gold_counts)}


def candidate_overlap(a: dict, b: dict, depth: int) -> dict:
    counts = [len(set(a[sid][:depth]) & set(b[sid][:depth])) for sid in a]
    return {"depth": depth, **distribution(counts)}


def recovered_displaced(samples: dict, control: dict, variant: dict) -> dict:
    recovered = displaced = 0
    for sid, sample in samples.items():
        gold = {str(x) for x in sample["answer"]}; base = set(control[sid][:100]); alt = set(variant[sid][:100])
        recovered += len((gold & alt) - base); displaced += len((gold & base) - alt)
    return {"hybrid_recovered_gold_documents": recovered, "hybrid_displaced_gold_documents": displaced}


def union_ce_records(samples: dict, variants: dict, chunks: list[dict]) -> tuple[list, dict]:
    supports = {name: support_map_from_candidates(candidates) for name, candidates in variants.items()}
    records, seen = [], set()
    for name, candidates in variants.items():
        for sid, sample in samples.items():
            for document in candidates[sid]:
                doc_id = document["document_id"]
                for idx in supports[name][sid][doc_id]:
                    key = ("ce", sid, doc_id, idx)
                    if key not in seen:
                        records.append((key, sample["question"], chunks[idx]["text"])); seen.add(key)
    return records, supports


In [ ]:

run_started = perf_counter()
dev, split_info = load_fixed_dev(LEGALIR_SOURCE_PATH)
documents = load_corpus(CORPUS_PATH)
chunks, chunking = fixed_window_chunks(documents, CHUNK_SIZE, CHUNK_OVERLAP)
if len(documents) != EXPECTED_DOCUMENTS or len(chunks) != EXPECTED_FIXED_CHUNKS:
    raise RuntimeError("fixed corpus mismatch")
dense = load_dense_model(); dense_metadata = dict(dense["metadata"])
corpus_encoding = encode_normalized_cls(dense, [x["text"] for x in chunks], CORPUS_BATCH_SIZE)
query_encoding = encode_normalized_cls(dense, [x["question"] for x in dev.values()], QUERY_BATCH_SIZE)
dense_corpus_encoding_seconds = corpus_encoding["seconds"]; dense_query_encoding_seconds = query_encoding["seconds"]
dense_retrieval = retrieve_dense_hits(query_encoding["embeddings"], corpus_encoding["embeddings"], chunks, list(dev))
bm25_retrieval = retrieve_bm25_hits(chunks, dev)
rrf_hits = fuse_hits(dense_retrieval["hits"], bm25_retrieval["hits"], "rrf")
score_hits = fuse_hits(dense_retrieval["hits"], bm25_retrieval["hits"], "normalized_score")
hit_variants = {"dense": dense_retrieval["hits"], "bm25": bm25_retrieval["hits"], "rrf": rrf_hits, "score_fusion": score_hits}
candidate_depth_info = {name: candidate_depth_diagnostics(hits, 200) for name, hits in hit_variants.items()}
candidates200 = {
    name: candidates_from_hits(hits, "sum_top2", 200, require_exact_depth=False)
    for name, hits in hit_variants.items()
}
if min(len(documents) for values in candidates200.values() for documents in values.values()) < 100:
    raise RuntimeError("a retrieval arm has fewer than 100 unique documents; fixed top-100 control unavailable")
rankings200 = {name: rankings_from_candidates(value) for name, value in candidates200.items()}
retrieval_metrics = {name: evaluate_rankings(dev, ranks, depths=(10, 20, 50, 100, 200)) for name, ranks in rankings200.items()}
assert_dense_baseline(retrieval_metrics["dense"])
candidates100 = {name: {sid: docs[:100] for sid, docs in values.items()} for name, values in candidates200.items()}
dense["model"].to("cpu"); del corpus_encoding, query_encoding; gc.collect(); torch.cuda.empty_cache()
reranker = load_reranker(); reranker_metadata = dict(reranker["metadata"])
records, supports = union_ce_records(dev, candidates100, chunks)
scored = score_pair_records(reranker, records)
ce_rankings = {name: derive_ce_ranking(dev, candidates, supports[name], scored["cache"], "sum_top2", "ce") for name, candidates in candidates100.items()}
final_metrics = {name: evaluate_rankings(dev, ranking) for name, ranking in ce_rankings.items()}
assert_m8_control(final_metrics["dense"])
contributions = {name: top5_contributions(dev, ranks) for name, ranks in ce_rankings.items()}
bootstraps = {}
for name in ("rrf", "score_fusion"):
    improved = final_metrics[name]["precision"] > final_metrics["dense"]["precision"] and final_metrics[name]["recall"] > final_metrics["dense"]["recall"]
    bootstraps[name] = paired_bootstrap(contributions["dense"], contributions[name]) if improved else {"ran": False, "reason": "variant did not improve both final precision and recall", "seed": BOOTSTRAP_SEED, "resamples": BOOTSTRAP_RESAMPLES}
result = {
    "experiment_name": "bm25_bge_hybrid_fusion_dev", "split": split_info, "source_sha256": EXPECTED_SOURCE_SHA256, "query_count": len(dev),
    "research_question": "Was hybrid search rejected prematurely by testing an uncapped document union instead of fixed-budget chunk fusion?",
    "research_axis": "chunk-level lexical+dense fixed-budget fusion", "control": "BGE-M3 dense only",
    "independent_variable": {"variants": ["equal-weight RRF k=60", "equal-weight per-query min-max score fusion"], "alpha_sweep": False},
    "fixed_components": {"chunking": "2000/200", "top_k_chunks_per_branch": 2000, "fused_chunk_budget": 2000, "document_score": "sum top2", "candidate_budget": 100, "support": "m8 from same variant top2000 chunk pool", "reranker": RERANKER_MODEL_NAME, "ce_document_score": "select top2 then sum", "final_k": 5},
    "models": {"dense": dense_metadata, "reranker": reranker_metadata}, "bm25": {"version": bm25s.__version__, "method": "lucene", "k1": 1.5, "b": 0.75, "tokenization": "lowercase Unicode \\w+; no stemming; no stopwords"},
    "chunking": chunking, "chunk_pool_gold_coverage": {name: pool_gold_coverage(dev, hits) for name, hits in hit_variants.items()},
    "retrieval_metrics": retrieval_metrics, "candidate_depth_diagnostics": candidate_depth_info,
    "gold_complementarity": {name: {str(k): complementarity(dev, rankings200["dense"], rankings200[name], k) for k in (100, 200)} for name in ("bm25", "rrf", "score_fusion")},
    "candidate_overlap_at_100": {name: candidate_overlap(rankings200["dense"], rankings200[name], 100) for name in ("bm25", "rrf", "score_fusion")},
    "candidate_overlap_at_200": {name: candidate_overlap(rankings200["dense"], rankings200[name], 200) for name in ("bm25", "rrf", "score_fusion")},
    "hybrid_gold_changes_at_100": {name: recovered_displaced(dev, rankings200["dense"], rankings200[name]) for name in ("rrf", "score_fusion")},
    "final_metrics": final_metrics, "final_delta_vs_dense": {name: metric_delta(final_metrics[name], final_metrics["dense"]) for name in ("rrf", "score_fusion")},
    "paired_behavior": {name: paired_behavior(contributions["dense"], contributions[name]) for name in ("rrf", "score_fusion")}, "paired_bootstrap": bootstraps,
    "runtime": {"dense_corpus_encoding_seconds": dense_corpus_encoding_seconds, "dense_query_encoding_seconds": dense_query_encoding_seconds, "dense_retrieval_seconds": dense_retrieval["seconds"], "bm25_build_and_retrieval_seconds": bm25_retrieval["seconds"], "ce": scored["diagnostics"], "total_seconds": perf_counter() - run_started, "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated())},
    "interpretation": "DEV-only controlled fusion evidence; no automatic holdout or public promotion."
}
save_result(result)
